# Introduction to Fama-French Factors

NQC S1, Week 5

First, we have to import the python libraries we need. Run the code below in order to do so.

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import statistics

Now, lets build a simple portfolio that we will use for this tutorial.

In [ ]:
portfolio_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

def load_returns(tickers):
    portfolio_returns = {}

    for ticker in tickers:
        returns = yf.download(ticker, start="2025-09-20", end="2026-09-21", auto_adjust=True, progress=False)
        close_price = returns["Close"]
        stock_returns = close_price.pct_change(fill_method=None).dropna(how="all")

        portfolio_returns[ticker] = stock_returns.squeeze()

    return portfolio_returns

portfolio = load_returns(portfolio_tickers)

Try printing the dictionary to see if everything worked, after running the code snippet above.

In [ ]:
print(load_returns(portfolio_tickers))

# Now, lets look into some different Fama-French factors.

The three more common Fama-French factors are:
- MKT - rf: market excess returns
- SMB: Small minus Big
- HML: High minus Low

These factor return series describe different patterns in stock returns. The regression coefficients will measure our portfolio's exposure to them.

The model is usually written as:

$$
R_i - R_f
=
\alpha_i
+
\beta_M (R_M - R_f)
+
\beta_{SMB} SMB
+
\beta_{HML} HML
+
\varepsilon_i
$$

We will use the S&P 500 universe for these portfolios, hence, we will need a list of all tickers, and to load the S&P 500's daily returns. Assume you got these from Bloomberg, in reality I asked chatgpt for the list.

First, lets use the function we just created to get the returns for each individual S&P 500 component, and the SPY index itself.

We will use SPY as our market proxy. With `auto_adjust=True`, its adjusted prices account for dividends and stock splits.

These are simplified teaching factors built within our S&P 500 sample. "Small" means smaller within this sample, rather than the full US small-cap market. We use current company characteristics to explain historical returns, so this exercise is not a point-in-time strategy backtest.

In [4]:
sp500_tickers = ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BE', 'BNY', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CTVA', 'CSGP', 'COST', 'CRH', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 'DDOG', 'DVA', 'DECK', 'DE', 'DELL', 'DAL', 'DVN', 'DXCM', 'FANG', 'DLR', 'DG', 'DLTR', 'D', 'DPZ', 'DASH', 'DOV', 'DOW', 'DHI', 'DTE', 'DUK', 'DD', 'ETN', 'EBAY', 'ECHO', 'ECL', 'EIX', 'EW', 'ELV', 'EME', 'EMR', 'ETR', 'EOG', 'EQT', 'EFX', 'EQIX', 'ERIE', 'ESS', 'EL', 'EG', 'EVRG', 'P', 'ES', 'EXC', 'EXE', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FDS', 'FICO', 'FAST', 'FRT', 'FDX', 'FDXF', 'FERG', 'FIS', 'FITB', 'FSLR', 'FE', 'FISV', 'FLEX', 'F', 'FTNT', 'FTV', 'FOXA', 'FOX', 'BEN', 'FCX', 'GRMN', 'IT', 'GE', 'GEHC', 'GEV', 'GEN', 'GNRC', 'GD', 'GIS', 'GM', 'GPC', 'GILD', 'GPN', 'GL', 'GDDY', 'GS', 'HAL', 'HIG', 'HAS', 'HCA', 'DOC', 'HSIC', 'HSY', 'HPE', 'HLT', 'HD', 'HONA', 'HON', 'HRL', 'HST', 'HWM', 'HPQ', 'HUBB', 'HUM', 'HBAN', 'HII', 'IBM', 'IEX', 'IDXX', 'ITW', 'ILMN', 'INCY', 'IR', 'PODD', 'INTC', 'IBKR', 'ICE', 'IFF', 'IP', 'INTU', 'ISRG', 'IVZ', 'INVH', 'IQV', 'IRM', 'JBHT', 'JBL', 'JKHY', 'J', 'JNJ', 'JCI', 'JPM', 'KVUE', 'KDP', 'KEY', 'KEYS', 'KMB', 'KIM', 'KMI', 'KKR', 'KLAC', 'KHC', 'KR', 'LHX', 'LH', 'LRCX', 'LVS', 'LDOS', 'LEN', 'LII', 'LLY', 'LIN', 'LYV', 'LMT', 'L', 'LOW', 'LULU', 'LITE', 'LYB', 'MTB', 'MPC', 'MAR', 'MRSH', 'MLM', 'MRVL', 'MAS', 'MA', 'MKC', 'MCD', 'MCK', 'MDT', 'MRK', 'META', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MRNA', 'MDLZ', 'MPWR', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'NDAQ', 'NTAP', 'NFLX', 'NEM', 'NWSA', 'NWS', 'NEE', 'NKE', 'NI', 'NDSN', 'NSC', 'NTRS', 'NOC', 'NCLH', 'NRG', 'NUE', 'NVDA', 'NVR', 'NXPI', 'ORLY', 'OXY', 'ODFL', 'OMC', 'ON', 'OKE', 'ORCL', 'OTIS', 'PCAR', 'PKG', 'PLTR', 'PANW', 'PSKY', 'PH', 'PAYX', 'PYPL', 'PNR', 'PEP', 'PFE', 'PCG', 'PM', 'PSX', 'PNW', 'PNC', 'PPG', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PTC', 'PSA', 'PHM', 'PWR', 'QCOM', 'DGX', 'Q', 'RL', 'RJF', 'RDDT', 'RTX', 'O', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RVTY', 'HOOD', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'SPGI', 'CRM', 'SNDK', 'SBAC', 'SLB', 'STX', 'SRE', 'NOW', 'SHW', 'SPG', 'SWKS', 'SJM', 'SW', 'SNA', 'SOLV', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'STLD', 'STE', 'SYK', 'SMCI', 'SYF', 'SNPS', 'SYY', 'TMUS', 'TROW', 'TTWO', 'TPR', 'TRGP', 'TGT', 'TEL', 'TDY', 'TER', 'TSLA', 'TXN', 'TPL', 'TXT', 'TMO', 'TJX', 'TKO', 'TSCO', 'TT', 'TDG', 'TRV', 'TRMB', 'TFC', 'TYL', 'TSN', 'USB', 'UBER', 'UDR', 'ULTA', 'UNP', 'UAL', 'UPS', 'URI', 'UNH', 'UHS', 'VLO', 'VEEV', 'VTR', 'VLTO', 'VRSN', 'VRSK', 'VZ', 'VRTX', 'VRT', 'VTRS', 'VICI', 'V', 'VST', 'VMRK', 'VMC', 'WRB', 'GWW', 'WAB', 'WMT', 'DIS', 'WBD', 'WM', 'WAT', 'WEC', 'WFC', 'WELL', 'WST', 'WDC', 'WY', 'WSM', 'WMB', 'WTW', 'WDAY', 'WYNN', 'XEL', 'XYL', 'YUM', 'ZBRA', 'ZBH', 'ZTS']

sp500_component_returns = load_returns(sp500_tickers)
spy_returns = load_returns(['SPY'])

Now, we can create the SMB portfolio.

Stocks above the median market cap are shorted, and the ones below are bought. Within each side, stocks are weighted in proportion to that side's total market cap. The long weights sum to +1 and the short weights sum to -1.

In [ ]:
def market_cap_weighting(tickers):

    sp500_market_cap = {}

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        market_cap = stock.fast_info["market_cap"]

        if market_cap is not None:
            sp500_market_cap[ticker] = market_cap

    median_cap = statistics.median(sp500_market_cap.values())

    small_caps = {}
    big_caps = {}

    for ticker in sp500_market_cap:
        if sp500_market_cap[ticker] > median_cap:
            big_caps[ticker] = sp500_market_cap[ticker]
        else:
            small_caps[ticker] = sp500_market_cap[ticker]

    weights = {}

    total_small_caps = sum(small_caps.values())
    total_big_caps = sum(big_caps.values())

    for ticker in big_caps:
        weights[ticker] = -big_caps[ticker] / total_big_caps

    for ticker in small_caps:
        weights[ticker] = small_caps[ticker] / total_small_caps

    return weights


sp500_smb_weights = market_cap_weighting(sp500_tickers)

Now that we have our simplified SMB portfolio, lets get out HML portfolio. HML simply compares high book-to-market stocks (value stocks) to low book-to-market stocks (growth stocks, such as Nvidia and AMD).

In [6]:
def hml_weights():

    sp500_btm = {}
    market_caps = {}

    for ticker in sp500_tickers:
        stock = yf.Ticker(ticker)

        mtb = stock.info.get("priceToBook")
        market_cap = stock.fast_info["market_cap"]

        if mtb is not None and mtb > 0 and market_cap is not None:
            btm = 1 / mtb

            sp500_btm[ticker] = btm
            market_caps[ticker] = market_cap

    median_btm = statistics.median(sp500_btm.values())

    growth_stocks = {}
    value_stocks = {}

    growth_caps = {}
    value_caps = {}

    for ticker in sp500_btm:
        if sp500_btm[ticker] > median_btm:
            value_stocks[ticker] = sp500_btm[ticker]
            value_caps[ticker] = market_caps[ticker]

        else:
            growth_stocks[ticker] = sp500_btm[ticker]
            growth_caps[ticker] = market_caps[ticker]

    weights = {}

    total_growth_cap = sum(growth_caps.values())
    total_value_cap = sum(value_caps.values())

    for ticker in value_stocks:
        weights[ticker] = value_caps[ticker] / total_value_cap

    for ticker in growth_stocks:
        weights[ticker] = -growth_caps[ticker] / total_growth_cap

    return weights


sp500_hml_weights = hml_weights()

Now that we have the weights for each portfolio, we must compute the daily returns for these portfolios. We already have a dictionary with the daily returns of each S&P500 component, and the weights for the SMB and HML porttfolios.

In [ ]:
def compute_returns(weights, returns):
    portfolio_returns = {}

    long_tickers = [ticker for ticker in weights if weights[ticker] > 0]
    short_tickers = [ticker for ticker in weights if weights[ticker] < 0]

    tickers = long_tickers + short_tickers

# Start with the first stock's available dates.
    dates = returns[tickers[0]].index

# Narrow this down to dates shared by every remaining stock.
    for ticker in tickers[1:]:
        dates = dates.intersection(returns[ticker].index)

    for day in dates:
        long_return = 0
        short_return = 0

        for ticker in long_tickers:
            long_return += weights[ticker] * returns[ticker].loc[day]

        for ticker in short_tickers:
            short_return += weights[ticker] * returns[ticker].loc[day]

        daily_return = long_return + short_return

        portfolio_returns[day] = daily_return

    return portfolio_returns

smb_returns = compute_returns(sp500_smb_weights, sp500_component_returns)
hml_returns = compute_returns(sp500_hml_weights, sp500_component_returns)

# Now, lets combine our factor returns

In [ ]:
smb_series = pd.Series(smb_returns, name="SMB")
hml_series = pd.Series(hml_returns, name="HML")
market_series = spy_returns['SPY']

# we convert to a series simply because it is much more convenient

def build_factor_data(market, smb, hml):
    market = market.rename("Market")
    smb = smb.rename("SMB")
    hml = hml.rename("HML")

    factor_data = pd.concat(
        [market, smb, hml],
        axis=1
    )

    factor_data = factor_data.dropna()

    return factor_data

factor_data = build_factor_data(
    market_series,
    smb_series,
    hml_series
)

print(factor_data.head())

              Market       SMB       HML
2026-06-16 -0.005962  0.003167  0.012098
2026-06-17 -0.012488 -0.008186  0.002559
2026-06-18  0.010397 -0.005782 -0.020906
2026-06-22 -0.003147  0.007476  0.016783
2026-06-23 -0.014522  0.012661  0.021043


# Regressing our investment portfolio

Now, lets use Market, SMB and HML to explain the returns of the five-stock investment portfolio we chose at the start.

We will give each stock an equal weight of 20%, kept constant each day. This is our teaching baseline. We have not downloaded a risk-free return series, so we assume a daily risk-free return of zero. Our intercept depends on this assumption.

First, we calculate the investment portfolio's returns and match every observation to the same date.

In [2]:
import statsmodels.api as sm

portfolio_weights = {}

for ticker in portfolio_tickers:
    portfolio_weights[ticker] = 1 / len(portfolio_tickers)

investment_returns = compute_returns(portfolio_weights, portfolio)
investment_series = pd.Series(investment_returns, name="Portfolio")

regression_data = pd.concat(
    [investment_series, factor_data],
    axis=1
)

regression_data = regression_data.dropna().sort_index()

risk_free_daily = 0.0

regression_data["Portfolio"] = regression_data["Portfolio"] - risk_free_daily
regression_data["Market"] = regression_data["Market"] - risk_free_daily

print("Number of observations:", len(regression_data))
print("First date:", regression_data.index.min().date())
print("Last date:", regression_data.index.max().date())

Number of observations: 66
First date: 2026-06-16
Last date: 2026-09-18


Lets estimate our three-factor regression. We add a constant so that the model can estimate an intercept, which is our daily alpha.

In the output, `Market`, `SMB` and `HML` are factor exposures. For example, a negative HML coefficient suggests that our portfolio behaves more like growth stocks, holding the other factors constant. These coefficients are sensitivities, rather than percentages of the portfolio invested in each style.

In [3]:
def run_regression(returns, factors):
    factors = sm.add_constant(factors)
    model = sm.OLS(returns, factors).fit()

    return model


three_factor_model = run_regression(
    regression_data["Portfolio"],
    regression_data[["Market", "SMB", "HML"]]
)

print("Three-factor coefficients:")
print(three_factor_model.params)

Three-factor coefficients:
const     0.000591
Market    1.029625
SMB       0.008805
HML      -0.445921
dtype: float64


# Assignment: weighting and model comparison

Now, it is your turn. Use the same five stocks, dates and factor portfolios from the lesson. Change the investment portfolio's weights, then compare how well CAPM and our three-factor model explain its returns.

Submit this notebook with your completed code, plots, results table and a short written interpretation. Keep SMB and HML unchanged. Continue using the lesson's zero risk-free-rate assumption.

## 1. Replace equal weights with market-cap weights

Build a **long-only** investment portfolio whose weights are proportional to the five companies' market capitalisations. Use one snapshot of market caps and keep these target weights constant each day, as in the lesson.

- Collect market caps for `portfolio_tickers`, using the same `yfinance` approach as the lesson.
- Require a positive, valid market cap for every stock. If one is unavailable, resolve the missing data before proceeding.
- Create one positive weight per stock and check that the weights sum to 1.
- Explain in one or two sentences why market-cap weighting suits this exercise's objective. Equal weighting remains a valid alternative for a different objective.

**Watch out:** `market_cap_weighting()` builds the SMB long/short portfolio. It splits companies at the median and creates negative weights, so it cannot be reused unchanged for this long-only investment portfolio.

In [ ]:
def investment_market_cap_weights(tickers):
    weights = {}

    # TODO: collect a valid market cap for each selected stock.
    # TODO: calculate positive weights proportional to market cap.

    return weights


# TODO: create assignment_weights by calling your function.
# TODO: check the stock names, positive weights and total weight.

## 2. Compute returns and align the dates

Use `compute_returns()` with your new investment weights and the existing `portfolio` dictionary. Convert the result into a Series named `Portfolio`.

Create `assignment_data` by combining that Series with the existing `factor_data`. Drop missing rows once and sort the dates. Apply `risk_free_daily` to the investment and market return columns, following the lesson. Use this same table for both models.

In [ ]:
# TODO: calculate assignment_returns using compute_returns().
# TODO: create assignment_series, named "Portfolio".
# TODO: combine it with factor_data to create assignment_data.
# TODO: drop missing observations, sort dates and apply the risk-free assumption.
# TODO: print the observation count and first and last dates.

## 3. Estimate CAPM and the three-factor model

Reuse `run_regression()` to create:

- `assignment_capm_model`, using only `Market` as an explanatory variable.
- `assignment_three_factor_model`, using `Market`, `SMB` and `HML`.

Both models must explain the reweighted investment portfolio, include an intercept and use exactly the same observations. Check their observation counts with `.nobs`.

In [ ]:
# TODO: fit assignment_capm_model.
# TODO: fit assignment_three_factor_model.
# TODO: confirm that both regressions use the same dates and observation count.

## 4. Plot both regressions

**CAPM:** Plot daily market excess returns on the horizontal axis and daily investment portfolio excess returns on the vertical axis. Add the fitted regression line using the estimated intercept and market beta. Sort the market returns before drawing the line.

**Model comparison:** Create two actual-versus-fitted scatter plots, one for CAPM and one for the three-factor model. Use `.fittedvalues` for fitted returns. Put fitted returns on the horizontal axis and actual returns on the vertical axis. Add an `actual = fitted` diagonal and use identical axis limits in both plots.

Label the axes, titles and units. If you display returns in percent, multiply both actual and fitted returns by 100. The three-factor fit combines all three factors, so a single line against the market alone does not represent the full model.

In [ ]:
import matplotlib.pyplot as plt

# TODO: draw the CAPM scatter plot and fitted regression line.
# TODO: draw actual-versus-fitted plots for both models.
# TODO: add the diagonal reference, matching scales and clear labels.

## 5. Compare R-squared

Create a table with one row per model and columns for **R-squared** and **observations**. You can access R-squared through `.rsquared`.

Write a short paragraph answering:

1. Which model explains more of this portfolio's historical return variation, and by how many percentage points does R-squared differ?
2. Why can adding SMB and HML increase R-squared? With the same observations and an intercept, adding regressors cannot lower ordinary in-sample R-squared.
3. Why does a higher R-squared alone not establish better predictions on future data?

R-squared measures the share of return variation explained in this sample. A value of 0.80 means 80%, rather than an 80% forecast accuracy.

**Optional:** Add daily alpha to the table using the intercept. Label its units and discuss how it changes between the two models. A positive alpha alone does not establish investment skill.

In [ ]:
assignment_results = {}

# TODO: collect R-squared and the observation count for each model.
# TODO: turn assignment_results into a readable comparison table.
# OPTIONAL: include daily alpha with clearly labelled units.

### Your interpretation

Weighting rationale:

*Write your explanation here.*

Model comparison:

*Write your paragraph here.*